# Digit addition: a from-scratch transformer, and its addition circuit

## What this project is

I train a small transformer from scratch on **decimal addition**, then reverse-engineer the
algorithm it learns.

This extends Nanda et al.'s *Progress measures for grokking via mechanistic interpretability*,
which analysed **modular** arithmetic and found the model had learned a Fourier algorithm built
out of trigonometric identities. Ordinary multi-digit addition is a different problem. It is
positional and it requires carrying, so whatever circuit appears should be structurally
different. That difference is the contribution.

## Why it is set up this way

Notebook 1 (`1_transformer_from_scratch.ipynb`) is the precursor: a GPT-2-shaped transformer
built by hand and trained on TinyStories. This notebook uses TransformerLens'
`HookedTransformer` for the model instead, so the hook points needed for the interpretability
section come for free and there is no `state_dict` mapping to debug later.

The important change is the **vocabulary**: 12 symbols (`0`-`9`, `+`, `=`) instead of 50257. In
notebook 1, 97% of the parameters were the embedding table and only 3% did any computation.
Here that inverts, and essentially every parameter computes. That is what makes the
interpretability tractable. There is an actual algorithm inside rather than a lookup table.

## The minimum result

**Identify which attention heads implement the carry, via ablation and attention patterns.**

Anything past that is bonus: the full mechanism, how circuits differ across digit positions,
grokking dynamics. This phase has no natural end, so the floor matters.

In [ ]:
from dataclasses import dataclass

import einops
import torch as t
from jaxtyping import Int
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print("device:", device)

---

## 1. Task encoding

- **Vocabulary**: `0`-`9`, `+` and `=`, so 12 symbols. A plain dict rather than a trained
  tokenizer.
- **Sequence format**: `49+97=641`, fixed length, so every example has the same shape.
  The answer is **reversed**, least significant digit first, so carries propagate in the same
  direction as generation. Writing it forwards would ask the model to predict the leading digit
  before it has computed the carries that determine it.
- **Zero-padding**: `05` rather than `5`, so a given position means the same thing in every
  example.
- **Number of digits**: 2. Enough to require carrying, small enough to enumerate exhaustively
  at 100 x 100 = 10,000 pairs.

In [ ]:
VOCABULARY = {f"{i:d}": i for i in range(10)}
VOCABULARY["+"] = 10
VOCABULARY["="] = 11

# convenience aliases
TOKEN_TO_ID = VOCABULARY
ID_TO_TOKEN = {idx: token for token, idx in TOKEN_TO_ID.items()}
D_VOCAB = len(VOCABULARY)

# 1D (seq,) or 2D (batch, seq) ids, as a tensor or plain lists
TokenIds = Tensor | list[int] | list[list[int]]


def to_tokens(strings: str | list[str]) -> Int[Tensor, "batch seq"]:
    """'49+97=641' -> tensor([[4, 9, 10, 9, 7, 11, 6, 4, 1]]). Always returns (batch, seq)."""
    if isinstance(strings, str):
        strings = [strings]
    return t.tensor([[TOKEN_TO_ID[c] for c in s] for s in strings], device=device)


def to_str_tokens(tokens: TokenIds) -> list[str] | list[list[str]]:
    """Token ids -> the characters they stand for, one per token. Nested if the input is 2D."""
    if isinstance(tokens, Tensor):
        tokens = tokens.tolist()
    if tokens and isinstance(tokens[0], list):
        return [[ID_TO_TOKEN[i] for i in row] for row in tokens]
    return [ID_TO_TOKEN[i] for i in tokens]


def to_string(tokens: TokenIds) -> str | list[str]:
    """Token ids -> '49+97=641'. A list of strings if the input is 2D."""
    strs = to_str_tokens(tokens)
    if strs and isinstance(strs[0], list):
        return ["".join(row) for row in strs]
    return "".join(strs)


# sequences will have the result of the computation reversed


## 2. Dataset

Every `(a, b)` pair is enumerated and then split. 10,000 examples of 9 tokens fits in memory as
a single tensor, so there is no need for `datasets`, Arrow, or DataLoader workers.

**The split fraction is the most consequential knob.** Holding out a large fraction is what
makes **grokking** possible: the model memorises the training pairs, sits near chance on the
held-out ones, then generalises abruptly thousands of steps later. Reproducing that on this
task would be a result in itself, so the fraction is worth sweeping.

Loss is computed **only on the answer positions**. The question digits are given, so predicting
them is free and would dilute the metric.

In [ ]:
def sum_dataset(num_digits: int = 2, seed: int = 42, frac_train: float = 0.3) -> tuple[Tensor, Tensor]:
    
    pairs = []

    for i in range(10**num_digits):
        for j in range(10**num_digits):
            pairs.append(f"{i:0{num_digits}d}+{j:0{num_digits}d}={f'{(i+j):0{num_digits+1}d}'[::-1]}")

    data = t.tensor([[TOKEN_TO_ID[token] for token in example] for example in pairs], device=device)
    g = t.Generator().manual_seed(seed)

    _n = data.shape[0]
    split = t.randperm(_n, generator=g)
    train_idx = split[:int(frac_train * _n)].to(device=device)
    test_idx = split[int(frac_train * _n):].to(device=device)


    assert t.cat([train_idx, test_idx], dim=0).sort().values.equal(t.arange(len(data), device=data.device))

    train = data[train_idx]
    test = data[test_idx]
    
    return train, test
train, test = sum_dataset()
print(f"train: {train.shape} \ntest: {test.shape}")

## 3. Train

Sizing follows Nanda et al. as closely as the different task allows: `d_model=128`, 4 heads,
`d_mlp=512`, and no layer norm. The deliberate differences are 2 layers rather than 1, a
12-symbol vocabulary, and a 9-token context, all of which follow from multi-digit addition
being a longer and more compositional task than single-step modular arithmetic.

Training is full batch with AdamW, a constant `lr=1e-3`, and `weight_decay=1.0`. That
configuration is not incidental. The grokking result depends on the decoupled weight decay in
particular, and full batch training keeps the loss curves clean enough for the transition to be
legible.

### Training

I started from ARENA's `TrainArgs` and trimmed it. Fields that duplicated `cfg` went, since two
sources of truth for the same number will eventually disagree, and so did the fields that only
make sense for `DataLoader` batching. With full batch training one epoch is one optimizer step,
so a single step counter is all that is needed.

In [ ]:
@dataclass
class TrainArgs:
    num_digits: int
    # trainset_size: int   -- derivable from train.shape[0]
    # valset_size: int     -- derivable from test.shape[0]
    frac_train: float
    epochs: int
    # batch_size: int      -- full batch, so there is nothing to vary
    lr: float
    # lr_start: float
    # lr_end: float
    weight_decay: float
    seed: int
    eval_every: int
    # d_model: int
    # d_head: int
    # n_layers: int
    # n_heads: int
    # d_mlp: int
    # normalization_type: str | None
    use_wandb: bool
    device: str

    def __post_init__(self):
        self.seq_len = self.num_digits * 3 + 2 + 1  # We have [{a}, +, {b}, =, {a+b}]

In [ ]:
args = TrainArgs(
    num_digits=2,
    frac_train=0.3,
    epochs=40000,
    lr=1e-3,
    weight_decay=1.0,
    seed=42,
    eval_every=10,
    use_wandb=False,
    device=device,
)

### Defining the model

In [ ]:
def create_model(args: TrainArgs) -> HookedTransformer:
    """A fresh model from init, sized from `args`.

    Re-seeded on every call, so a sweep over `frac_train` varies only the thing
    being swept -- every run starts from identical weights.
    """
    t.manual_seed(args.seed)

    d_model = 128
    n_heads = 4

    cfg = HookedTransformerConfig(
        n_layers=2,
        n_ctx=args.seq_len,  # We have [{a}, +, {b}, =, {a+b}]
        d_model=d_model,
        d_head=d_model // n_heads,
        n_heads=n_heads,
        d_mlp=4 * d_model,
        attn_only=False,
        act_fn="relu",
        # We have all digits, and "+="
        d_vocab=len(VOCABULARY),
        # it's a small transformer so may as well use these hooks
        use_attn_result=True,
        use_split_qkv_input=True,
        use_hook_tokens=True,
        # Layernorm makes things way more accurate, even though it makes
        # mech interp a little more annoying!
        normalization_type=None,
        device=args.device,
    )

    return HookedTransformer(cfg)


model = create_model(args)
cfg = model.cfg

In [ ]:
def param_breakdown(model: HookedTransformer, n_train: int | None = None) -> None:
    """Parameter counts grouped by component, with the attention:MLP ratio.

    The ratio is worth watching more than the total -- attention routes information
    between positions, MLPs do the arithmetic, and a config that starves the MLPs
    will struggle on this task. The standard d_mlp = 4 * d_model gives ~0.50x.
    """
    groups: dict[str, int] = {}
    for name, param in model.named_parameters():
        key = "attn" if ".attn." in name else "mlp" if ".mlp." in name else "embed/unembed"
        groups[key] = groups.get(key, 0) + param.numel()

    total = sum(groups.values())
    for key, count in sorted(groups.items(), key=lambda kv: -kv[1]):
        print(f"  {key:16s} {count:>10,}  ({count / total * 100:5.1f}%)")
    print(f"  {'total':16s} {total:>10,}")

    if groups.get("mlp"):
        print(f"\n  attn:mlp ratio   {groups['attn'] / groups['mlp']:.2f}x")
    if n_train:
        print(f"  params per training example   {total / n_train:.0f}")


param_breakdown(model)


In [ ]:
for name, param in model.named_parameters():
    print(f"{name:<20} {str(param.shape):<20} {param.numel()}")
sum(p.numel() for p in model.parameters())      # 584,556

In [ ]:
print(
    train.shape,
    to_string(train[t.randint(low=0, high=len(train), size=(1,))]),
    sep="\n",
)

In [ ]:
class Trainer:
    def __init__(self, model: HookedTransformer, args: TrainArgs):
        self.args = args
        self.model = model
        
    def training_step(self, toks: Tensor) -> Tensor:
        logits, target = self._shared_train_validation_step(toks)
        return self._loss(logits, target)
    
    def validation_step(self, toks: Tensor) -> tuple[float, float]:
        logits, target = self._shared_train_validation_step(toks)
        loss = self._loss(logits, target).item()
        accuracy = (logits.argmax(dim=-1) == target).float().mean().item()
        return loss, accuracy
    
    def _loss(self, logits: Tensor, target: Tensor) -> Tensor:
        return t.nn.functional.cross_entropy(
            einops.rearrange(logits, "batch seq vocab_out -> (batch seq) vocab_out"),
            einops.rearrange(target, "batch seq -> (batch seq)")
        )
    
    def _shared_train_validation_step(self, toks: Tensor) -> tuple[Tensor, Tensor]:
        toks = toks.to(self.args.device)
        logits = self.model(toks)[:, -(self.args.num_digits + 2): -1]
        target = toks[:, -(self.args.num_digits + 1):]
        return logits, target
    
    def configure_optimizers(self):
        optimizer = t.optim.AdamW(
            self.model.parameters(), lr=self.args.lr, weight_decay=self.args.weight_decay
        )
        return optimizer

def train_model(model: HookedTransformer, args: TrainArgs):
    trainer = Trainer(model=model, args=args)
    optimizer = trainer.configure_optimizers()
    
    train, test = sum_dataset(
        num_digits=args.num_digits, seed=args.seed, frac_train=args.frac_train
    )

    # the two curves that make the grokking plot, plus test accuracy
    history = {"step": [], "train_loss": [], "test_loss": [], "test_accuracy": []}

    # training
    optimizer.zero_grad()
    progress_bar = tqdm(range(args.epochs))
    for epoch in progress_bar:
        loss = trainer.training_step(train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        # validation
        if epoch % args.eval_every == 0 or epoch == args.epochs - 1:
            with t.inference_mode():
                test_loss, test_accuracy = trainer.validation_step(test)
            history["step"].append(epoch)
            history["train_loss"].append(loss.item())
            history["test_loss"].append(test_loss)
            history["test_accuracy"].append(test_accuracy)
            progress_bar.set_description(
                f"train {loss.item():.4f} | test {test_loss:.4f} | acc {test_accuracy:.3f}"
            )

    return trainer.model, history


## 4. Interpretability

Planned, roughly in order of cost:

- **Attention patterns** on carry versus non-carry examples, via `circuitsvis`. Do specific
  heads look at the digit positions that generate a carry?
- **Per-head ablation**: zero `hook_z` for each head in turn and measure the accuracy drop.
  This identifies *which* heads matter, not *how* they work.
- **Direct logit attribution**: which component actually writes the answer digit.
- **The mechanism**: how carry information moves between digit positions.

Method reference is ARENA's exercise on the Nanda result, *1.5.2 Grokking & Modular
Arithmetic*. (Hook names and shapes: see my annotated diagram in `notes/`, which lives outside
this repo.)

**Attention sinks are the likely false positive here.** These sequences have no BOS token, so a
head with nothing useful to attend to will dump its attention on whatever sits at position 0,
or on the `=`. A head that appears to attend to the equals sign for no clear reason is probably
a sink rather than part of a circuit.

## 5. Write-up

Goes in `README.md` and `results/`: motivation, method, findings, plots. The central comparison
is the circuit found here against Nanda's modular-arithmetic one. That contrast is the point of
the project.